In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-02-27 23:54:30 | people | execute | Started
2025-02-27 23:54:30 | people | load | Started
2025-02-27 23:54:35 | people | load | Completed in 0.08 min
2025-02-27 23:54:35 | people | transform | Started
2025-02-27 23:54:35 | people | transform | Completed in 0.0 min
2025-02-27 23:54:35 | people | write | Started
2025-02-27 23:55:47 | people | write | Completed in 1.2 min
2025-02-27 23:55:47 | people | execute | Completed in 1.28 min
2025-02-27 23:55:47 | planets | execute | Started
2025-02-27 23:55:47 | planets | load | Started
2025-02-27 23:55:50 | planets | load | Completed in 0.03 min
2025-02-27 23:55:50 | planets | transform | Started
2025-02-27 23:55:50 | planets | transform | Completed in 0.0 min
2025-02-27 23:55:50 | planets | write | Started
2025-02-27 23:56:53 | planets | write | Completed in 1.03 min
2025-02-27 23:56:53 | planets | execute | Completed in 1.08 min


In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-02-27 23:54:...|        Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-02-27 23:54:...|  Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-02-27 23:54:...|    Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-02-27 23:54:...|      Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-02-27 23:54:...|              Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-02-27 23:54:...|              Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-02-27 23:54:...|Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-02-27 23:54:...|         Jango Fett| 69|https://www.swapi...|{"created": "2025...|
|2025-02

In [11]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(truncate=False)

No. Rows: 60
+--------------------------+--------------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+--------------+---+--

In [12]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [13]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [14]:
# with load filter and transformation
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df


silver_instance = StarWarsSilver(spark, **options)

In [15]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people", "planets"
)

2025-02-27 23:56:57 | people | execute | Started
2025-02-27 23:56:57 | people | load | Started
2025-02-27 23:56:57 | people | load | Completed in 0.0 min
2025-02-27 23:56:57 | people | transform | Started
2025-02-27 23:56:57 | people | transform | Completed in 0.0 min
2025-02-27 23:56:57 | people | write | Started
2025-02-27 23:57:00 | people | write | Completed in 0.03 min
2025-02-27 23:57:00 | people | execute | Completed in 0.03 min
2025-02-27 23:57:00 | planets | execute | Started
2025-02-27 23:57:00 | planets | load | Started
2025-02-27 23:57:00 | planets | load | Completed in 0.0 min
2025-02-27 23:57:00 | planets | transform | Started
2025-02-27 23:57:00 | planets | transform | Completed in 0.0 min
2025-02-27 23:57:00 | planets | write | Started
2025-02-27 23:57:01 | planets | write | Completed in 0.02 min
2025-02-27 23:57:01 | planets | execute | Completed in 0.02 min


In [16]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 17
+--------------------------+--------------------------+---------------------+---+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|url                                 |properties                                                                                                                                                                                                                |
+--------------------------+--------------------------+---------------------+---+------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 18
+-------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS              |LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                       |
+-------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2025-02-27 23:57:00.

In [18]:
# without load filter
silver_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-02-27 23:57:04 | people | execute | Started
2025-02-27 23:57:04 | people | load | Started
2025-02-27 23:57:04 | people | load | Completed in 0.0 min
2025-02-27 23:57:04 | people | transform | Started
2025-02-27 23:57:04 | people | transform | Completed in 0.0 min
2025-02-27 23:57:04 | people | write | Started
2025-02-27 23:57:05 | people | write | Completed in 0.02 min
2025-02-27 23:57:05 | people | execute | Completed in 0.02 min
2025-02-27 23:57:05 | planets | execute | Started
2025-02-27 23:57:05 | planets | load | Started
2025-02-27 23:57:05 | planets | load | Completed in 0.0 min
2025-02-27 23:57:05 | planets | transform | Started
2025-02-27 23:57:05 | planets | transform | Completed in 0.0 min
2025-02-27 23:57:05 | planets | write | Started
2025-02-27 23:57:07 | planets | write | Completed in 0.02 min
2025-02-27 23:57:07 | planets | execute | Completed in 0.02 min


In [19]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-02-27 23:57:...|2025-02-27 23:54:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|               C-3PO|  2|https://www.swapi...|{167, 75, n/a, go...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|               R2-D2|  3|https://www.swapi...|{96, 32, n/a, whi...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|         Darth Vader|  4|https://www.swapi...|{202, 136, none, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|         Leia Organa|  5|https://www.swapi...|{150, 49, brown, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|           Owen Lars|  6|https://www.swapi...|{178, 120,

In [20]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 60
+--------------------+--------------------+-----------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|       name|uid|                 url|          properties|
+--------------------+--------------------+-----------+---+--------------------+--------------------+
|2025-02-27 23:57:...|2025-02-27 23:55:...|   Mon Cala| 31|https://www.swapi...|{11030, 21, 398, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|  Chandrila| 32|https://www.swapi...|{13500, 20, 368, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|    Sullust| 33|https://www.swapi...|{12780, 20, 263, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|   Toydaria| 34|https://www.swapi...|{7900, 21, 184, 1...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|  Malastare| 35|https://www.swapi...|{18880, 26, 201, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|   Dathomir| 36|https://www.swapi...|{10480, 24, 491, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|     Ryloth| 37|https://ww

# 3 Replace Where

In [21]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid > '0'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df

    def get_replace_condition(self, df: DataFrame, table: str) -> str:
        return "uid > 0"


silver_instance = StarWarsSilver(spark, **options)
silver_instance.load(filter="custom").transform().write(mode="replace").execute(
    "people", "planets"
)

2025-02-27 23:57:09 | people | execute | Started
2025-02-27 23:57:09 | people | load | Started
2025-02-27 23:57:09 | people | load | Completed in 0.0 min
2025-02-27 23:57:09 | people | transform | Started
2025-02-27 23:57:09 | people | transform | Completed in 0.0 min
2025-02-27 23:57:09 | people | write | Started
2025-02-27 23:57:12 | people | write | Completed in 0.03 min
2025-02-27 23:57:12 | people | execute | Completed in 0.03 min
2025-02-27 23:57:12 | planets | execute | Started
2025-02-27 23:57:12 | planets | load | Started
2025-02-27 23:57:12 | planets | load | Completed in 0.0 min
2025-02-27 23:57:12 | planets | transform | Started
2025-02-27 23:57:12 | planets | transform | Completed in 0.0 min
2025-02-27 23:57:12 | planets | write | Started
2025-02-27 23:57:14 | planets | write | Completed in 0.03 min
2025-02-27 23:57:14 | planets | execute | Completed in 0.03 min


In [22]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-02-27 23:57:...|2025-02-27 23:54:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|               C-3PO|  2|https://www.swapi...|{167, 75, n/a, go...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|               R2-D2|  3|https://www.swapi...|{96, 32, n/a, whi...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|         Darth Vader|  4|https://www.swapi...|{202, 136, none, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|         Leia Organa|  5|https://www.swapi...|{150, 49, brown, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|           Owen Lars|  6|https://www.swapi...|{178, 120,

# 4 Append

In [23]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("custom == 'custom'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df


silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write().execute(
    "people", "planets"
)  # default mode is append

2025-02-27 23:57:15 | people | execute | Started
2025-02-27 23:57:15 | people | load | Started
2025-02-27 23:57:15 | people | load | Completed in 0.0 min
2025-02-27 23:57:15 | people | transform | Started
2025-02-27 23:57:15 | people | transform | Completed in 0.0 min
2025-02-27 23:57:15 | people | write | Started
2025-02-27 23:57:17 | people | write | Completed in 0.02 min
2025-02-27 23:57:17 | people | execute | Completed in 0.02 min
2025-02-27 23:57:17 | planets | execute | Started
2025-02-27 23:57:17 | planets | load | Started
2025-02-27 23:57:17 | planets | load | Completed in 0.0 min
2025-02-27 23:57:17 | planets | transform | Started
2025-02-27 23:57:17 | planets | transform | Completed in 0.0 min
2025-02-27 23:57:17 | planets | write | Started
2025-02-27 23:57:18 | planets | write | Completed in 0.02 min
2025-02-27 23:57:18 | planets | execute | Completed in 0.02 min


In [24]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 164
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-02-27 23:57:...|2025-02-27 23:54:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|               C-3PO|  2|https://www.swapi...|{167, 75, n/a, go...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|               R2-D2|  3|https://www.swapi...|{96, 32, n/a, whi...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|         Darth Vader|  4|https://www.swapi...|{202, 136, none, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|         Leia Organa|  5|https://www.swapi...|{150, 49, brown, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|           Owen Lars|  6|https://www.swapi...|{178, 120

# 5 Merge

In [25]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df

    def get_delta_merge_builder(
        self, df: DataFrame, delta_table: DeltaTable
    ) -> DeltaMergeBuilder:
        merge_condition = "target.url = source.url"
        builder = delta_table.alias("target").merge(
            df.alias("source"), merge_condition
        )
        builder = builder.whenMatchedUpdateAll()
        builder = builder.whenNotMatchedInsertAll()
        return builder


silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write(mode="merge").execute("people", "planets")

2025-02-27 23:57:19 | people | execute | Started
2025-02-27 23:57:19 | people | load | Started
2025-02-27 23:57:19 | people | load | Completed in 0.0 min
2025-02-27 23:57:19 | people | transform | Started
2025-02-27 23:57:19 | people | transform | Completed in 0.0 min
2025-02-27 23:57:19 | people | write | Started
2025-02-27 23:57:22 | people | write | Completed in 0.03 min
2025-02-27 23:57:22 | people | execute | Completed in 0.05 min
2025-02-27 23:57:22 | planets | execute | Started
2025-02-27 23:57:22 | planets | load | Started
2025-02-27 23:57:22 | planets | load | Completed in 0.0 min
2025-02-27 23:57:22 | planets | transform | Started
2025-02-27 23:57:22 | planets | transform | Completed in 0.0 min
2025-02-27 23:57:22 | planets | write | Started
2025-02-27 23:57:25 | planets | write | Completed in 0.03 min
2025-02-27 23:57:25 | planets | execute | Completed in 0.03 min


In [26]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 164
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-02-27 23:57:...|2025-02-27 23:54:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84, blond, ...|
|2025-02-27 23:57:...|2025-02-27 23:54:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84,

In [27]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 120
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|2025-02-27 23:57:...|2025-02-27 23:55:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-02-27 23:57:...|2025-02-27 23:55:..

# 6 Clean Up

In [28]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]